# Baseline Logistic Regression

**Notebook 04 — Phase 1**

## Executive Summary

Train the Phase 1 baseline using logistic regression only — an interpretable linear classifier appropriate for HR auditability.

## Objectives

- Explain why logistic regression is the Phase 1 choice
- Compare against a dummy baseline
- Train leakage-safe sklearn Pipeline
- Persist the fitted model

## Expected Outputs

- `models/phase1/baseline_pipeline.joblib`

## Required Inputs

- Run from the **project root** with the virtual environment activated (`make install`).
- Prerequisites: Notebook 03 (`data/interim/train_features.csv`, `test_features.csv`)
- Reproducibility: `RANDOM_STATE = 42` where applicable.


## 1. Why Logistic Regression?

**Advantages:** Interpretable coefficients, fast training, strong baseline for tabular HR data, probabilistic outputs for risk ranking.

**Limitations:** Linear decision boundary; cannot capture complex interactions (addressed in Phase 2).

## 2. Setup and Load Data

In [1]:
import sys
from pathlib import Path

_root = Path.cwd()
if not (_root / "data" / "raw").exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from notebooks._shared.config import INTERIM_DIR, MODEL_DIR, RANDOM_STATE, TARGET
from notebooks._shared.preprocessing import encode_targets, get_feature_columns

MODEL_DIR.mkdir(parents=True, exist_ok=True)
train_df = pd.read_csv(INTERIM_DIR / "train_features.csv")
test_df = pd.read_csv(INTERIM_DIR / "test_features.csv")

X_train = train_df.drop(columns=[TARGET])
y_train = encode_targets(train_df[TARGET])
X_test = test_df.drop(columns=[TARGET])
y_test = encode_targets(test_df[TARGET])

categorical_cols, numerical_cols = get_feature_columns(train_df, include_engineered=True)
print(f"Features: {len(categorical_cols)} categorical, {len(numerical_cols)} numerical")


def build_preprocessor(numerical_cols, categorical_cols):
    """StandardScaler + OneHotEncoder — fit on train only via Pipeline."""
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numerical_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
        ],
        remainder="drop",
    )


def evaluate_classifier(y_true, y_pred, y_prob, model_name="Logistic Regression"):
    """Compute standard binary classification metrics on the held-out test set."""
    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1 Score": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "Avg Precision": average_precision_score(y_true, y_prob),
    }
    return pd.DataFrame([metrics]).round(4)


Features: 9 categorical, 29 numerical


## 3. Dummy Baseline

**Objective:** Establish a naive benchmark.

**Why:** Quantifies lift from a real model.

In [2]:
dummy_pipeline = Pipeline([
    ("preprocessor", build_preprocessor(numerical_cols, categorical_cols)),
    ("classifier", DummyClassifier(strategy="most_frequent")),
])
dummy_pipeline.fit(X_train, y_train)
dummy_metrics = evaluate_classifier(
    y_test,
    dummy_pipeline.predict(X_test),
    dummy_pipeline.predict_proba(X_test)[:, 1],
    model_name="Dummy (most_frequent)",
)
dummy_metrics

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC,Avg Precision
0,Dummy (most_frequent),0.8401,0.0,0.0,0.0,0.5,0.1599


## 4. Implementation — Train Logistic Regression

**Methodology:** `ColumnTransformer` + `LogisticRegression` in a single Pipeline. Preprocessing fits on train only.

**Hyperparameters:** `C=1.0`, `class_weight='balanced'`, `max_iter=1000`.

In [3]:
pipeline = Pipeline([
    ("preprocessor", build_preprocessor(numerical_cols, categorical_cols)),
    ("classifier", LogisticRegression(
        penalty="l2",
        C=1.0,
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])

pipeline.fit(X_train, y_train)
model_path = MODEL_DIR / "baseline_pipeline.joblib"
joblib.dump(pipeline, model_path)

lr_metrics = evaluate_classifier(
    y_test,
    pipeline.predict(X_test),
    pipeline.predict_proba(X_test)[:, 1],
    model_name="Logistic Regression",
)
metrics_df = pd.concat([dummy_metrics, lr_metrics], ignore_index=True)
print(f"Model saved: {model_path}")
metrics_df

Model saved: /mnt/d/Abhishek/Employee-Attrition Project/models/phase1/baseline_pipeline.joblib


/mnt/d/Abhishek/Employee-Attrition Project/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC,Avg Precision
0,Dummy (most_frequent),0.8401,0.000,0.0000,0.0000,0.5000,0.1599
1,Logistic Regression,0.7721,0.381,0.6809,0.4885,0.8085,0.5249


### Observations

Logistic regression substantially outperforms the dummy classifier on ROC-AUC and recall.

**Business Interpretation:** The model captures meaningful attrition signal; detailed evaluation in Notebook 05.

**Conclusion:** Phase 1 baseline trained and persisted.

## Future Connection

Notebook 05 evaluates the persisted baseline on the held-out test set with confusion matrix, ROC/PR curves, coefficient analysis, and HR recommendations.
